# Seq2Seq English to French Translation

This notebook implements a basic sequence-to-sequence model using LSTM networks to translate English to French. It uses one-hot encoding for characters and builds an encoder-decoder model with teacher forcing.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense

In [9]:
# Runtime Settings
batch_limit = 64
epoch_limit = 100
latent_size = 256
sample_limit = 10000
data_file = '/content/drive/MyDrive/eng-french.txt'
model_output = 'eng2french.h5'

Data Preprocessing

In [10]:
source_sentences = []
target_sentences = []
src_vocab = set()
tgt_vocab = set()

In [11]:
try:
    with open(data_file, encoding='utf-8') as file:
        corpus = file.read().split('\n')
except FileNotFoundError:
    print(f"Missing file: {data_file}")
    exit()

In [12]:
for record in corpus[:min(sample_limit, len(corpus) - 1)]:
    try:
        src, tgt = record.split('\t')
    except ValueError:
        continue
    tgt = '\t' + tgt + '\n'
    source_sentences.append(src)
    target_sentences.append(tgt)
    src_vocab.update(src)
    tgt_vocab.update(tgt)

src_vocab = sorted(list(src_vocab))
tgt_vocab = sorted(list(tgt_vocab))

src_token_len = len(src_vocab)
tgt_token_len = len(tgt_vocab)
src_seq_maxlen = max(len(s) for s in source_sentences)
tgt_seq_maxlen = max(len(s) for s in target_sentences)

print(f"Samples loaded: {len(source_sentences)}")
print(f"Unique tokens (source): {src_token_len}")
print(f"Unique tokens (target): {tgt_token_len}")
print(f"Max source sequence length: {src_seq_maxlen}")
print(f"Max target sequence length: {tgt_seq_maxlen}")


Samples loaded: 10000
Unique tokens (source): 71
Unique tokens (target): 94
Max source sequence length: 16
Max target sequence length: 59


In [13]:
src_token_map = {char: idx for idx, char in enumerate(src_vocab)}
tgt_token_map = {char: idx for idx, char in enumerate(tgt_vocab)}

encoder_input = np.zeros((len(source_sentences), src_seq_maxlen, src_token_len), dtype='float32')
decoder_input = np.zeros((len(target_sentences), tgt_seq_maxlen, tgt_token_len), dtype='float32')
decoder_target = np.zeros((len(target_sentences), tgt_seq_maxlen, tgt_token_len), dtype='float32')


In [14]:
for idx, (src, tgt) in enumerate(zip(source_sentences, target_sentences)):
    for t, ch in enumerate(src):
        encoder_input[idx, t, src_token_map[ch]] = 1.
    if t + 1 < src_seq_maxlen:
        encoder_input[idx, t + 1:, src_token_map.get(' ', 0)] = 1.

    for t, ch in enumerate(tgt):
        decoder_input[idx, t, tgt_token_map[ch]] = 1.
        if t > 0:
            decoder_target[idx, t - 1, tgt_token_map[ch]] = 1.
    if t + 1 < tgt_seq_maxlen:
        decoder_input[idx, t + 1:, tgt_token_map.get(' ', 0)] = 1.
    if t < tgt_seq_maxlen:
        decoder_target[idx, t:, tgt_token_map.get(' ', 0)] = 1.

Model Construction

In [15]:
# Encoder
enc_input = Input(shape=(None, src_token_len))
enc_lstm = LSTM(latent_size, return_state=True)
enc_output, h_enc, c_enc = enc_lstm(enc_input)
enc_states = [h_enc, c_enc]

In [16]:
# Decoder
dec_input = Input(shape=(None, tgt_token_len))
dec_lstm = LSTM(latent_size, return_sequences=True, return_state=True)
dec_output, _, _ = dec_lstm(dec_input, initial_state=enc_states)
dec_dense = Dense(tgt_token_len, activation='softmax')
dec_output = dec_dense(dec_output)

In [17]:
# Seq2Seq model
translator = Model([enc_input, dec_input], dec_output)

Model Training

In [18]:
print("\n>>> Starting Training...")
translator.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
translator.fit([encoder_input, decoder_input], decoder_target,
               batch_size=batch_limit,
               epochs=epoch_limit,
               validation_split=0.2)

translator.save(model_output)
print(f"Model archived at: {model_output}")


>>> Starting Training...
Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.6887 - loss: 1.6377 - val_accuracy: 0.6874 - val_loss: 1.2099
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.7311 - loss: 1.0175 - val_accuracy: 0.6984 - val_loss: 1.0769
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7473 - loss: 0.9094 - val_accuracy: 0.7276 - val_loss: 0.9783
Epoch 4/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.7741 - loss: 0.8060 - val_accuracy: 0.7503 - val_loss: 0.8699
Epoch 5/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.7938 - loss: 0.7209 - val_accuracy: 0.7627 - val_loss: 0.8133
Epoch 6/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.8044 - loss: 0.6743 - val_accuracy: 0.7710 - val_loss: 0.7699
Epoch 7/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8115 - loss: 0.6455 - val_accuracy: 0.7809 - val_loss: 0.7454
Epoch 8/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy

Model archived at: eng2french.h5


Inference Model

In [19]:
print("\n>>> Creating Inference Modules...")

encoder_inf = Model(enc_input, enc_states)

h_inf = Input(shape=(latent_size,))
c_inf = Input(shape=(latent_size,))
dec_state_inputs = [h_inf, c_inf]
dec_output_inf, h_next, c_next = dec_lstm(dec_input, initial_state=dec_state_inputs)
dec_states = [h_next, c_next]
dec_output_inf = dec_dense(dec_output_inf)

decoder_inf = Model([dec_input] + dec_state_inputs, [dec_output_inf] + dec_states)

reverse_src_index = {idx: ch for ch, idx in src_token_map.items()}
reverse_tgt_index = {idx: ch for ch, idx in tgt_token_map.items()}


>>> Creating Inference Modules...


Decoding New Sequences

In [20]:
def translate_sequence(encoded_input):
    state_vals = encoder_inf.predict(encoded_input, verbose=0)
    target_seed = np.zeros((1, 1, tgt_token_len))
    target_seed[0, 0, tgt_token_map['\t']] = 1.

    translation = ''
    finished = False

    while not finished:
        output, h_out, c_out = decoder_inf.predict([target_seed] + state_vals, verbose=0)
        sampled_idx = np.argmax(output[0, -1, :])
        sampled_char = reverse_tgt_index[sampled_idx]
        translation += sampled_char

        if sampled_char == '\n' or len(translation) > tgt_seq_maxlen:
            finished = True

        target_seed = np.zeros((1, 1, tgt_token_len))
        target_seed[0, 0, sampled_idx] = 1.
        state_vals = [h_out, c_out]

    return translation

Example Predictions

In [21]:
print("\n>>> Example Predictions:")
for i in range(100):
    test_seq = encoder_input[i: i + 1]
    predicted = translate_sequence(test_seq)
    print("\nSource:", source_sentences[i])
    print("Translated:", predicted.strip())


>>> Example Predictions:

Source: Go.
Translated: Va !

Source: Run!
Translated: Avenez !

Source: Run!
Translated: Avenez !

Source: Wow!
Translated: Pous gager !

Source: Fire!
Translated: Venez à la maison.

Source: Help!
Translated: Venez.

Source: Jump.
Translated: Va es cleuche !

Source: Bye.
Translated: T'andez-vous !

Source: Stop!
Translated: Arrête !

Source: Stop!
Translated: Arrête !

Source: Stop!
Translated: Arrête !

Source: Wait!
Translated: Attendez !

Source: Wait!
Translated: Attendez !

Source: Go on.
Translated: Va !

Source: Go on.
Translated: Va !

Source: Go on.
Translated: Va !

Source: I see.
Translated: Je l'ai confrariée.

Source: I try.
Translated: Je l'ai enterdu.

Source: I won!
Translated: Je t'ai laissé tomber.

Source: I won!
Translated: Je t'ai laissé tomber.

Source: Oh no!
Translated: Est-ce que je fait cela ?

Source: Attack!
Translated: Attentez !

Source: Attack!
Translated: Attentez !

Source: Cheers!
Translated: Venez.

Source: Cheers!
Transl